# Autocw AI Customer Support Agent — Exploration Notebook

This notebook demonstrates:
1. **Brand Volume Analysis**: Exploring inbound/outbound volume across brands in `twcs.csv`.
2. **Thread Reconstruction**: Multi-turn dialogue graph traversal.
3. **Text Cleaning & Sanitization**: Twitter mentions, links, and entity removal.
4. **Intent Clustering & Taxonomy**: BERTopic / K-Means keyword analysis.
5. **Semantic Retrieval & RAG**: MiniLM vector search and response drafting.
6. **Hybrid Escalation**: Deterministic rules and borderline reasoning.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from src.config import PROJECT_ROOT, RAW_DATASET_FILE, CONVERSATIONS_FILE, QUERY_REPLY_PAIRS_FILE
from src.utils.text import clean_tweet_text
from src.pipeline import process_message

## 1. Brand Volume Analysis in `twcs.csv`

In [ ]:
if RAW_DATASET_FILE.exists():
    df = pd.read_csv(RAW_DATASET_FILE, nrows=500000, low_memory=False)
    outbound = df[df['inbound'] == False]
    brand_counts = outbound['author_id'].value_counts().head(10)
    print("Top 10 Support Handles by Outbound Volume (Sampled):")
    print(brand_counts)
else:
    print("twcs.csv not found at data/raw/twcs.csv")

## 2. Reconstructed Thread Inspection

In [ ]:
if CONVERSATIONS_FILE.exists():
    with open(CONVERSATIONS_FILE, 'r', encoding='utf-8') as f:
        sample_thread = json.loads(f.readline())
    
    print(f"Thread ID: {sample_thread['thread_id']}")
    print(f"Brand: {sample_thread['brand']}")
    print("\nDialogue Turns:")
    for i, turn in enumerate(sample_thread['turns'], 1):
        print(f"  [{i}] ({turn['role'].upper()} - {turn['author_id']}): {turn['text']}")

## 3. Query-Reply Semantic Retrieval

In [ ]:
from src.drafter.draft import retrieve_similar

test_query = "My package has been delayed for three days and tracking has not moved."
matches = retrieve_similar(test_query, top_k=3)
print(f"Query: {test_query}\n")
for idx, m in enumerate(matches, 1):
    print(f"--- Match {idx} (Similarity: {m['score']:.3f}) ---")
    print(f"Historical Customer: {m['query']}")
    print(f"Historical Agent:    {m['reply']}\n")

## 4. End-to-End Agent Execution

In [ ]:
queries = [
    "Where can I check the estimated arrival date of my order?",
    "I returned the defective boots last Monday, when will I get my money back?",
    "If you do not refund my money today, I am going to have my lawyer file a lawsuit!"
]

for q in queries:
    res = process_message(q)
    print("=" * 70)
    print(f"Customer: {q}")
    print(f"Intent:   {res['intent']} (confidence: {res['confidence']})")
    print(f"Escalate: {res['escalate']} (reason: {res['escalation_reason']})")
    print(f"Draft:    {res['draft_reply']}")